# 🎬 Media Downloader (Google Colab 雲端直存版)

**功能特色：**
- 🚀 **純雲端執行，本地 0 頻寬消耗**：由 Google 機房高速頻寬下載切片並於雲端完成 FFmpeg 封裝。
- ☁️ **原生直存 Google Drive**：影片下載合成後直接寫入掛載的 Google Drive，不需二次上傳。
- 🛡️ **智慧過濾**：自動過濾線上影片 (Movieffm / Gimy 等) 片頭與貼片賭博廣告。
- 🌐 **多平台支援**：Movieffm、Gimy、YouTube、Bilibili、Facebook、Instagram、TikTok、X/Twitter 等。

## 步驟 1：安裝依賴環境與掛載 Google Drive
執行下方儲存格安裝必要套件，並點擊授權掛載 Google Drive。

In [ ]:
#@title ⚙️ 1. 環境安裝與掛載 Google Drive
# 安裝解析與高速下載依賴 (Colab 預設已內建 FFmpeg)
!pip install -q yt-dlp curl_cffi

from google.colab import drive
import os

# 掛載 Google Drive
drive_mount_path = '/content/drive'
if not os.path.exists(drive_mount_path):
    drive.mount(drive_mount_path)
    print("✅ Google Drive 掛載成功！")
else:
    print("ℹ️ Google Drive 已在掛載狀態。")

## 步驟 2：載入核心下載引擎
若此筆記本在專案資料夾內運行，會直接讀取同目錄下的 `core_downloader.py`；若為獨立執行，將自動載入核心邏輯。

In [ ]:
#@title 📦 2. 載入核心下載引擎
import sys
if os.path.exists('core_downloader.py'):
    import core_downloader as cd
    print("✅ 成功載入本地 core_downloader.py 模組！")
elif os.path.exists('/content/core_downloader.py'):
    sys.path.append('/content')
    import core_downloader as cd
    print("✅ 成功載入 /content/core_downloader.py 模組！")
else:
    # 自動從 GitHub 讀取最新版核心引擎
    import urllib.request
    raw_url = 'https://raw.githubusercontent.com/hangout1997/Media-Download/main/core_downloader.py'
    print("正在由 GitHub 下載最新 core_downloader.py...")
    try:
        urllib.request.urlretrieve(raw_url, '/content/core_downloader.py')
        sys.path.append('/content')
        import core_downloader as cd
        print("✅ 成功自動下載並載入 core_downloader 核心引擎！")
    except Exception as e:
        print(f"⚠️ 由 GitHub 下載失敗 ({e})，請確認網路或手動上傳 core_downloader.py 至 Colab 檔案區。")

## 步驟 3：輸入影片網址並開始雲端直存下載
填寫下列參數並點擊執行，影片將由 Google 機房高速下載、去除廣告切片、無損封裝為 MP4 並**直接儲存至您的 Google Drive**！

In [ ]:
#@title 🚀 3. 開始雲端下載與直存任務
#@markdown ### 網址與儲存設定：
video_urls = "https://www.movieffm.net/movies/your-name/" #@param {type:"string"}
save_folder = "Download" #@param {type:"string"}
only_audio = False #@param {type:"boolean"}

#@markdown ### 選填 Cookie (登入/風控解鎖)：
fb_cookie = "" #@param {type:"string"}
bili_cookie = "" #@param {type:"string"}

import os, time, sys

# 設定 Google Drive 目標儲存路徑
target_dir = os.path.join('/content/drive/MyDrive', save_folder.strip('/'))
os.makedirs(target_dir, exist_ok=True)

urls = [u.strip() for u in video_urls.strip().split('\n') if u.strip()]
cookies = {'fb_cookie': fb_cookie.strip(), 'bili_cookie': bili_cookie.strip()}

print(f"📁 Google Drive 目標資料夾: {target_dir}")
print(f"🔗 待處理網址數量: {len(urls)} 個\n" + "="*60)

for idx, raw_url in enumerate(urls, 1):
    url = cd.normalize_input_url(raw_url)
    print(f"\n📍 [{idx}/{len(urls)}] 正在解析: {url}")
    
    try:
        items = cd.get_media_items(url, cookies=cookies)
        if not items:
            print(f"❌ 找不到可用的串流或影片來源: {url}")
            continue
            
        for item in items:
            title = item.get('title', 'video')
            ext = 'mp3' if only_audio else item.get('ext', 'mp4')
            filename = f"{title}.{ext}"
            out_file = os.path.join(target_dir, filename)
            
            if os.path.exists(out_file):
                print(f"⏭️ 雲端已存在同名檔案，自動跳過: {filename}")
                continue
                
            print(f"📥 開始下載並直存: {filename}")
            
            # 定義即時進度印出函式
            last_pct = [-1]
            def _progress(pct, cur_dur, tot_dur, speed_mb, speed_x):
                cur_p = int(pct * 100)
                if cur_p != last_pct[0] and cur_p % 5 == 0:
                    last_pct[0] = cur_p
                    print(f"\r⏳ 進度: {cur_p}% | 速度: {speed_mb:.2f} MB/s ({speed_x:.2f}x)", end="", flush=True)
                    
            res_path = cd.download_media_to_path(
                item,
                out_path=out_file,
                force_audio=only_audio,
                cookies=cookies,
                progress_cb=_progress
            )
            print(f"\n🎉 成功直存至 Google Drive: {res_path} ({os.path.getsize(res_path)/1024/1024:.1f} MB)")
            
    except Exception as e:
        print(f"\n❌ 下載失敗 ({url}): {e}")

print("\n" + "="*60 + "\n🎊 所有下載與直存任務處理完畢！可直接至 Google Drive 查看檔案。")